### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let\'s start by thinking about what I know. Parrots are known for mimicking human speech, right? But why do they do that? Maybe they use it to communicate with other parrots? Or maybe it\'s a way to bond with humans? I\'ve heard that some birds imitate sounds in their environment, like other birds or even machines. Is talking a learned behavior? \n\nWait, I remember something about parrots having a syrinx, which is their vocal organ. That might be more flexible than other birds, allowing them to produce a wider range of sounds. So maybe they can copy human speech because of their vocal anatomy. But why would they want to? Do they get rewards for talking? Like if a human talks to them or gives them treats, they might repeat it for the reward. That sounds like operant conditioning. \n\nAlso, some sources say that talking is a form of social behavior. Parrots are social animals, so in the wild, they probably commu

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
model_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017995C1C590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017995C1D2B0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. So I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'r08t7pjvs', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 154, 'total_tokens': 241, 'completion_time': 0.137056074, 'completion_tokens_details': {'reasoning_tokens': 63}, 'prompt_time': 0.007495076, 'prompt_tokens_details': None, 'queue_time': 0.047579883, 'total_time': 0.14455115}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019c940a-557d-72d3

In [5]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'r08t7pjvs',
  'type': 'tool_call'}]

In [6]:

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


In [5]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'g9cjqj927',
  'type': 'tool_call'}]

In [7]:
get_weather

StructuredTool(name='get_weather', description='Get the weather at a location', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x0000017995CEE020>)

In [7]:
get_weather.invoke(response.tool_calls[0])

ToolMessage(content="It's sunny in Boston", name='get_weather', tool_call_id='g9cjqj927')

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
messages


[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools provided. There\'s a function called get_weather that takes a location parameter. The required parameter is location, which should be a string. Since the user specified Boston, I need to call get_weather with location set to "Boston". I\'ll make sure the JSON is correctly formatted with the name of the function and the arguments as specified.\n', 'tool_calls': [{'id': 'vy8qg8943', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 153, 'total_tokens': 261, 'completion_time': 0.153810991, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.006371701, 'prompt_tokens_details': None, 'queue_time': 0.046081288, 'total_time': 0.160182692}, 'model_name

In [9]:
ai_msg.usage_metadata

{'input_tokens': 153,
 'output_tokens': 108,
 'total_tokens': 261,
 'output_token_details': {'reasoning': 84}}

In [7]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'vy8qg8943',
  'type': 'tool_call'}]

In [ ]:

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with location set to "Boston". Make sure the JSON is correctly formatted with the function name and arguments. No other functions are available, so this should be straightforward.\n', 'tool_calls': [{'id': '20cfda4p0', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 153, 'total_tokens': 262, 'completion_time': 0.199966855, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.006519155, 'prompt_tokens_details': None, 'queue_time': 0.055720395, 'total_time': 0.20648601}, 'mod